<a href="https://colab.research.google.com/github/dshrestha37/investment-research-agent/blob/main/News_Ingestion_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Real-Time Financial News Ingestion & FAISS Vector Pipeline

This pipeline pulls real-time stock news directly from Yahoo Finance via yfinance (requiring no API key) and optionally integrates with NewsAPI.org.

The pipeline ingests live headlines, cleans and chunks the text, tags each snippet with metadata (source, timestamp, URL), embeds the data into a FAISS vector database, and executes semantic retrieval with source citations for an auditable analyst brief.

In [1]:
# Install modular LangChain packages along with yfinance and FAISS
!pip install -qU yfinance langchain langchain-community langchain-core langchain-huggingface langchain-text-splitters sentence-transformers faiss-cpu transformers torch
!pip install --upgrade torch torchvision torchaudio

## Live Ingestion & FAISS Vector Database

Queries external news sources in real time using a specific stock ticker (e.g., *AAPL*). It extracts raw article data, including article titles, text summaries, publication dates, publisher names, and canonical source URLs.

In [2]:
# Install compatible packages
!pip install --upgrade torch torchvision
!pip install -qU transformers sentence-transformers faiss-cpu yfinance langchain-community langchain-core langchain-huggingface langchain-text-splitters

In [4]:
import datetime
import re
import yfinance as yf
from transformers import pipeline

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# CLASSIFICATION STRATEGIES

EARNINGS_KEYWORDS = [
    r"\bearnings\b", r"\brevenue\b", r"\beps\b", r"\bnet income\b",
    r"\bprofit\b", r"\bquarterly\b", r"\bfiscal\b", r"\bguidance\b",
    r"\b10-k\b", r"\b10-q\b", r"\boperating income\b", r"\bmargin\b",
    r"\bq1\b", r"\bq2\b", r"\bq3\b", r"\bq4\b", r"\bwall street expectations\b"
]

def is_earnings_keyword(text: str) -> bool:
    pattern = "|".join(EARNINGS_KEYWORDS)
    return bool(re.search(pattern, text, re.IGNORECASE))

print("Loading HuggingFace Zero-Shot Classifier model...")
nlp_classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
)

def classify_article_nlp(title: str, summary: str, confidence_threshold: float = 0.55) -> dict:
    text_sample = f"{title}. {summary}"[:512]
    candidate_labels = ["quarterly earnings and financial results", "general market news or corporate chatter"]

    res = nlp_classifier(text_sample, candidate_labels)
    top_label = res["labels"][0]
    score = res["scores"][0]

    is_earnings = (top_label == "quarterly earnings and financial results") and (score >= confidence_threshold)

    return {
        "is_earnings": is_earnings,
        "category": "earnings" if is_earnings else "general_noise",
        "confidence": round(score, 3)
    }

# INGESTION WITH FILTERING & METADATA TAGGING

def fetch_and_process_news(ticker_symbol: str, filter_mode: str = "tag_all"):
    print(f"\nFetching live news for {ticker_symbol}...")
    ticker = yf.Ticker(ticker_symbol)
    raw_news = ticker.news

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
    documents = []

    for item in raw_news:
        content = item.get("content", {})
        title = content.get("title", item.get("title", "No Title"))
        summary = content.get("summary", item.get("summary", ""))
        url = content.get("canonicalUrl", {}).get("url", item.get("link", ""))
        provider = content.get("provider", {}).get("displayName", item.get("publisher", "Yahoo Finance"))

        full_text = f"Title: {title}\nSummary: {summary}"

        classification = classify_article_nlp(title, summary)
        is_earnings = classification["is_earnings"]
        category = classification["category"]
        confidence = classification["confidence"]

        print(f"[{category.upper()}] (Score: {confidence}) -> {title[:65]}...")

        if filter_mode == "strict_earnings" and not is_earnings:
            continue

        chunks = text_splitter.split_text(full_text)
        for chunk in chunks:
            doc = Document(
                page_content=chunk,
                metadata={
                    "title": title,
                    "url": url,
                    "publisher": provider,
                    "category": category,
                    "is_earnings": is_earnings,
                    "classification_confidence": confidence
                }
            )
            documents.append(doc)

    print(f"\nTotal indexed chunks from API: {len(documents)}")
    return documents

# BUILD FAISS INDEX & RUN QUERY

TICKER = "AAPL"

# Fetch & process articles
processed_docs = fetch_and_process_news(TICKER, filter_mode="tag_all")

# SAFETY CHECK (Prevents FAISS crash if Yahoo blocks Colab)

if len(processed_docs) == 0:
    print("\n[WARNING] Yahoo Finance returned 0 articles (Colab IP was rate-limited).")
    print("Injecting fallback financial articles so FAISS pipeline continues...\n")

    processed_docs = [
        Document(
            page_content="Apple reported a record quarterly revenue of $85.8 billion, up 5% year-over-year. The strong quarterly earnings beat Wall Street expectations, driven by higher services revenue and solid iPhone sales.",
            metadata={"title": "Apple Q3 Earnings Beat Expectations", "url": "https://finance.yahoo.com/news/apple-q3", "publisher": "Yahoo Fallback", "category": "earnings", "is_earnings": True, "classification_confidence": 0.98}
        ),
        Document(
            page_content="Analysts expect Apple to highlight new artificial intelligence integrations across iOS during upcoming developer events to sustain market momentum.",
            metadata={"title": "Apple Prepares New AI Features", "url": "https://finance.yahoo.com/news/apple-ai", "publisher": "Yahoo Fallback", "category": "general_noise", "is_earnings": False, "classification_confidence": 0.85}
        )
    ]


print("\nBuilding FAISS vector index...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(processed_docs, embedding_model)
print("FAISS Index Ready!")

# FILTERED RETRIEVAL (Audit Trail Ready)

def retrieve_filtered_news(query: str, earnings_only: bool = True, top_k: int = 3):
    fetch_k = top_k * 3 if earnings_only else top_k
    results = vector_db.similarity_search(query, k=fetch_k)

    filtered_results = []
    for doc in results:
        if earnings_only and not doc.metadata.get("is_earnings", False):
            continue
        filtered_results.append(doc)
        if len(filtered_results) == top_k:
            break

    context_blocks = []
    audit_trail = []

    for i, res in enumerate(filtered_results):
        context_blocks.append(f"[Chunk {i+1}]\n{res.page_content}")
        meta = res.metadata
        audit_trail.append(
            f"Source [{i+1}]: '{meta['title']}' | Category: {meta['category']} | Confidence: {meta['classification_confidence']} | URL: {meta['url']}"
        )

    return "\n\n".join(context_blocks), "\n".join(audit_trail)

# Run test query
query = "What were the latest quarterly revenue and profit numbers?"
context, audit = retrieve_filtered_news(query, earnings_only=True, top_k=2)

print("\n" + "="*50)
print("FILTERED CONTEXT (EARNINGS ONLY):")
print("="*50)
print(context)

print("\n" + "="*50)
print("PROVENANCE & METADATA AUDIT:")
print("="*50)
print(audit)

Loading HuggingFace Zero-Shot Classifier model...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


Fetching live news for AAPL...

Total indexed chunks from API: 0

[WARNING] Yahoo Finance returned 0 articles (Colab IP was rate-limited).
Injecting fallback financial articles so FAISS pipeline continues...


Building FAISS vector index...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS Index Ready!

FILTERED CONTEXT (EARNINGS ONLY):
[Chunk 1]
Apple reported a record quarterly revenue of $85.8 billion, up 5% year-over-year. The strong quarterly earnings beat Wall Street expectations, driven by higher services revenue and solid iPhone sales.

PROVENANCE & METADATA AUDIT:
Source [1]: 'Apple Q3 Earnings Beat Expectations' | Category: earnings | Confidence: 0.98 | URL: https://finance.yahoo.com/news/apple-q3


## Earnings & Financial News Classification Engine for FAISS

This introduces an automated classification step before chunking and indexing.

It provides two complementary approaches:

**Keyword/Heuristic Classifier:**

Zero-latency, rule-based filtering using financial and earnings regex patterns.

**Zero-Shot NLP Classifier:**

 A contextual machine learning model (using HuggingFace transformers) that analyzes semantic intent to distinguish financial/earnings news from general noise.

In [7]:
# Install libraries
!pip install -qU yfinance langchain-core langchain-community langchain-huggingface langchain-text-splitters sentence-transformers faiss-cpu transformers torch

## Classification Pipeline & FAISS Indexing

In [8]:
import datetime
import re
import yfinance as yf
from transformers import pipeline

# Modern LangChain Imports (Prevents ModuleNotFoundError & Deprecation Warnings)
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# CLASSIFICATION STRATEGIES

EARNINGS_KEYWORDS = [
    r"\bearnings\b", r"\brevenue\b", r"\beps\b", r"\bnet income\b",
    r"\bprofit\b", r"\bquarterly\b", r"\bfiscal\b", r"\bguidance\b",
    r"\b10-k\b", r"\b10-q\b", r"\boperating income\b", r"\bmargin\b",
    r"\bq1\b", r"\bq2\b", r"\bq3\b", r"\bq4\b", r"\bwall street expectations\b"
]

def is_earnings_keyword(text: str) -> bool:
    """Fast regex check to detect financial keywords."""
    pattern = "|".join(EARNINGS_KEYWORDS)
    return bool(re.search(pattern, text, re.IGNORECASE))

print("Loading HuggingFace Zero-Shot Classifier model...")
nlp_classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
)

def classify_article_nlp(title: str, summary: str, confidence_threshold: float = 0.55) -> dict:
    """Classifies an article into earnings/financial vs general noise."""
    text_sample = f"{title}. {summary}"[:512]
    candidate_labels = ["quarterly earnings and financial results", "general market news or corporate chatter"]

    res = nlp_classifier(text_sample, candidate_labels)
    top_label = res["labels"][0]
    score = res["scores"][0]

    is_earnings = (top_label == "quarterly earnings and financial results") and (score >= confidence_threshold)

    return {
        "is_earnings": is_earnings,
        "category": "earnings" if is_earnings else "general_noise",
        "confidence": round(score, 3)
    }

# INGESTION & PROCESSING

def fetch_and_process_news(ticker_symbol: str, filter_mode: str = "tag_all"):
    """Fetches live news, classifies each article, and creates chunks with metadata."""
    print(f"\nFetching live news for {ticker_symbol}...")
    ticker = yf.Ticker(ticker_symbol)
    raw_news = ticker.news

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
    documents = []

    if not raw_news:
        return documents

    for item in raw_news:
        content = item.get("content", {})
        title = content.get("title", item.get("title", "No Title"))
        summary = content.get("summary", item.get("summary", ""))
        url = content.get("canonicalUrl", {}).get("url", item.get("link", ""))
        provider = content.get("provider", {}).get("displayName", item.get("publisher", "Yahoo Finance"))

        full_text = f"Title: {title}\nSummary: {summary}"

        classification = classify_article_nlp(title, summary)
        is_earnings = classification["is_earnings"]
        category = classification["category"]
        confidence = classification["confidence"]

        print(f"[{category.upper()}] (Score: {confidence}) -> {title[:65]}...")

        if filter_mode == "strict_earnings" and not is_earnings:
            continue

        chunks = text_splitter.split_text(full_text)
        for chunk in chunks:
            doc = Document(
                page_content=chunk,
                metadata={
                    "title": title,
                    "url": url,
                    "publisher": provider,
                    "category": category,
                    "is_earnings": is_earnings,
                    "classification_confidence": confidence
                }
            )
            documents.append(doc)

    print(f"\nTotal indexed chunks from API: {len(documents)}")
    return documents

# FAISS INDEX BUILD (WITH FALLBACK SAFETY)

TICKER = "AAPL"

# Process articles
processed_docs = fetch_and_process_news(TICKER, filter_mode="tag_all")

# Safety check: Prevent FAISS crash if Yahoo blocks Colab IP
if len(processed_docs) == 0:
    print("\n[WARNING] Yahoo Finance API returned 0 articles (Colab IP rate-limited).")
    print("Injecting fallback financial data to build FAISS index safely...\n")

    processed_docs = [
        Document(
            page_content="Apple reported a record quarterly revenue of $85.8 billion, up 5% year-over-year. The strong quarterly earnings beat Wall Street expectations, driven by higher services revenue and solid iPhone sales.",
            metadata={"title": "Apple Q3 Earnings Beat Expectations", "url": "https://finance.yahoo.com/news/apple-q3", "publisher": "Yahoo Fallback", "category": "earnings", "is_earnings": True, "classification_confidence": 0.98}
        ),
        Document(
            page_content="Analysts expect Apple to highlight new artificial intelligence integrations across iOS during upcoming developer events to sustain market momentum.",
            metadata={"title": "Apple Prepares New AI Features", "url": "https://finance.yahoo.com/news/apple-ai", "publisher": "Yahoo Fallback", "category": "general_noise", "is_earnings": False, "classification_confidence": 0.85}
        )
    ]

print("Building FAISS vector index...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(processed_docs, embedding_model)
print("FAISS Index Ready!")

# FILTERED RETRIEVAL & CITATIONS

def retrieve_filtered_news(query: str, earnings_only: bool = True, top_k: int = 3):
    """Retrieves chunks from FAISS and prints citations for auditability."""
    fetch_k = top_k * 3 if earnings_only else top_k
    results = vector_db.similarity_search(query, k=fetch_k)

    filtered_results = []
    for doc in results:
        if earnings_only and not doc.metadata.get("is_earnings", False):
            continue
        filtered_results.append(doc)
        if len(filtered_results) == top_k:
            break

    # Fall back to all top results if no strictly marked earnings chunks are found
    if not filtered_results and results:
        filtered_results = results[:top_k]

    context_blocks = []
    audit_trail = []

    for i, res in enumerate(filtered_results):
        context_blocks.append(f"[Chunk {i+1}]\n{res.page_content}")
        meta = res.metadata
        audit_trail.append(
            f"Source [{i+1}]: '{meta['title']}' | Category: {meta.get('category', 'N/A')} | Confidence: {meta.get('classification_confidence', 'N/A')} | URL: {meta.get('url', 'N/A')}"
        )

    return "\n\n".join(context_blocks), "\n".join(audit_trail)

# Run test query
query = "What were the latest quarterly revenue and profit numbers?"
context, audit = retrieve_filtered_news(query, earnings_only=True, top_k=2)

print("\n" + "="*50)
print("FILTERED CONTEXT (FOR LLM PROMPT):")
print("="*50)
print(context)

print("\n" + "="*50)
print("PROVENANCE & METADATA AUDIT:")
print("="*50)
print(audit)

Loading HuggingFace Zero-Shot Classifier model...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


Fetching live news for AAPL...

[WARNING] Yahoo Finance API returned 0 articles (Colab IP rate-limited).
Injecting fallback financial data to build FAISS index safely...

Building FAISS vector index...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS Index Ready!

FILTERED CONTEXT (FOR LLM PROMPT):
[Chunk 1]
Apple reported a record quarterly revenue of $85.8 billion, up 5% year-over-year. The strong quarterly earnings beat Wall Street expectations, driven by higher services revenue and solid iPhone sales.

PROVENANCE & METADATA AUDIT:
Source [1]: 'Apple Q3 Earnings Beat Expectations' | Category: earnings | Confidence: 0.98 | URL: https://finance.yahoo.com/news/apple-q3


## Saving & Reloading FAISS via LangChain

In [9]:
# SAVE THE FAISS INDEX TO DISK

SAVE_PATH = "faiss_news_index"

# Save index bin file and pkl metadata store
vector_db.save_local(SAVE_PATH)
print(f"FAISS vector store saved locally to directory: '{SAVE_PATH}'")


# RELOAD THE FAISS INDEX

from langchain_community.vectorstores import FAISS

loaded_vector_db = FAISS.load_local(
    folder_path=SAVE_PATH,
    embeddings=embedding_model,
    allow_dangerous_deserialization=True
)

print("FAISS vector store successfully reloaded from disk!")

FAISS vector store saved locally to directory: 'faiss_news_index'
FAISS vector store successfully reloaded from disk!


## Persistence & Incremental Index Management

Saves the binary FAISS vector index (**index.faiss**) and document metadata mapping (**index.pkl**) directly to disk or Google Drive.

It also includes functions to reload existing indexes in future sessions and incrementally append new news chunks without re-embedding or rebuilding the entire vector store from scratch.

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


import os

# Define path inside Google Drive
drive_save_path = "/content/drive/MyDrive/investment_research_agent/faiss_news_index"

# Save to Google Drive
vector_db.save_local(drive_save_path)
print(f"FAISS index permanently saved to Google Drive at: {drive_save_path}")

Files Created by save_local()

Inside the saved folder (faiss_news_index), LangChain creates two files:

**index.faiss:**

The binary FAISS vector index containing the dense numerical embeddings.

**index.pkl:**

A serialized Python dictionary holding the chunk text content and metadata (URLs, publishers, dates, classification scores).

In [13]:
import os
from google.colab import drive
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Mount Google Drive
drive.mount('/content/drive')

# Re-initialize embedding model (using modern import)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Define path and check existence
drive_save_path = "/content/drive/MyDrive/investment_research_agent/faiss_news_index"

if os.path.exists(drive_save_path):
    vector_db = FAISS.load_local(
        folder_path=drive_save_path,
        embeddings=embedding_model,
        allow_dangerous_deserialization=True
    )
    print("FAISS index successfully loaded from Google Drive!")
else:
    print("No existing FAISS index found at path. Run the ingestion pipeline first.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

No existing FAISS index found at path. Run the ingestion pipeline first.
